# Reinforcement Learning on FrozenLake: DQN vs PPO

## Author
- Michael Brackin (CPSC 425 AI, Dominican University)

---

# Project Description

This project explores reinforcement learning by training AI agents to solve the FrozenLake environment from Gymnasium. The objective is to compare how different reinforcement learning algorithms perform in a stochastic environment where movement is slippery and uncertain.

The project implements and evaluates two reinforcement learning approaches:

1. Deep Q-Network (DQN)
2. Proximal Policy Optimization (PPO)

Both agents are trained to navigate from the start state to the goal while avoiding holes in the environment.

---

# Problem Statement

FrozenLake is a grid-based environment where the agent must safely navigate across frozen tiles to reach a goal state. The environment becomes difficult because movement is stochastic (slippery), meaning actions do not always lead to the intended direction.

The challenge is to train reinforcement learning agents that can learn an optimal navigation policy despite uncertainty and sparse rewards.

---

# Solution Approach

## DQN Approach
The DQN implementation uses:
- One-hot encoded state representation
- Neural network for Q-value approximation
- Experience replay memory
- Epsilon-greedy exploration strategy

## PPO Approach
The PPO implementation uses:
- Stable-Baselines3 PPO agent
- Policy optimization methods
- Monitored training environment

## Reward Shaping
To improve learning efficiency, reward shaping was added to the DQN training loop:
- Small negative reward for each step (-0.005)
- Larger negative reward (-0.05) for ending an episode without reaching the goal
- Positive reward for reaching the goal (+1)

This helps reduce random wandering and encourages efficient paths.

---

# Environment Information

Environment:
- FrozenLake-v1
- Gymnasium RL environment
- 4x4 map
- Slippery environment enabled

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import matplotlib.pyplot as plt

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True, render_mode="rgb_array")

In [ ]:
state_size = env.observation_space.n
action_size = env.action_space.n

def one_hot(state, size):
    vec = np.zeros(size)
    vec[state] = 1
    return vec


In [ ]:
class DQN(nn.Module):
    def __init__(self):
        super(DQN, self).__init__()
        self.fc1 = nn.Linear(state_size, 32)
        self.fc2 = nn.Linear(32, action_size)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

model = DQN()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()


In [ ]:
memory = deque(maxlen=4000)

In [ ]:
#variables
gamma = 0.99
epsilon = 1.0
epsilon_decay = 0.995
min_epsilon = 0.01
batch_size = 32

episodes = 5000
rewards = []

In [ ]:
#runtime
for episode in range(episodes):
    state, _ = env.reset()
    state = one_hot(state, state_size)
    total_reward = 0
    done = False

    while not done:
        # Choose action
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            with torch.no_grad():
                q_values = model(torch.FloatTensor(state))
                action = torch.argmax(q_values).item()

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        if reward == 0:
          reward = -0.005

        if done and reward == -0.005:
          reward = -0.05
        next_state_oh = one_hot(next_state, state_size)

        # Store experience
        memory.append((state, action, reward, next_state_oh, done))
        state = next_state_oh
        total_reward += reward

        # Train from memory
        if len(memory) >= batch_size:
            batch = random.sample(memory, batch_size)

            states, actions, rewards_b, next_states, dones = zip(*batch)

            states = torch.FloatTensor(states)
            next_states = torch.FloatTensor(next_states)
            rewards_b = torch.FloatTensor(rewards_b)
            dones = torch.FloatTensor(dones)

            q_values = model(states)
            next_q_values = model(next_states)

            target = q_values.clone()

            for i in range(batch_size):
                if dones[i]:
                    target[i][actions[i]] = rewards_b[i]
                else:
                    target[i][actions[i]] = rewards_b[i] + gamma * torch.max(next_q_values[i])

            loss = criterion(q_values, target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    epsilon = max(min_epsilon, epsilon * epsilon_decay)
    rewards.append(total_reward)

In [ ]:
plt.plot(np.convolve(rewards, np.ones(100)/100, mode='valid'))
plt.title("DQN Average Reward")
plt.show()

In [ ]:
!pip install imageio imageio-ffmpeg -qq

In [ ]:
import imageio

num_render_episodes = 1 # We'll render one episode as a GIF

for episode in range(num_render_episodes):
    state, _ = env.reset()
    state = one_hot(state, state_size)
    done = False
    frames = []
    print(f"Creating GIF for Episode {episode + 1}")

    while not done:
        frames.append(env.render())
        with torch.no_grad():
            action = torch.argmax(model(torch.FloatTensor(state))).item()

        next_state, reward, terminated, truncated, _ = env.step(action)
        state = one_hot(next_state, state_size)
        done = terminated or truncated

    frames.append(env.render()) # Add final state
    imageio.mimsave(f'frozen_lake_dqn_episode_{episode+1}.gif', frames, fps=10)
    print(f"GIF saved as frozen_lake_dqn_episode_{episode+1}.gif")

env.close()

In [ ]:
successes = 0
for _ in range(1000):
    state, _ = env.reset()
    state = one_hot(state, state_size)
    done = False

    while not done:
        with torch.no_grad():
            action = torch.argmax(model(torch.FloatTensor(state))).item()
        state, reward, terminated, truncated, _ = env.step(action)
        state = one_hot(state, state_size)
        done = terminated or truncated

    successes += reward

print("DQN Success rate:", successes / 1000)

In [ ]:
pip install stable-baselines3 gymnasium

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback

# New environment for PPO
ppo_env = gym.make("FrozenLake-v1", is_slippery=True)
ppo_env = Monitor(ppo_env)

# PPO model (IMPORTANT: new variable name)
ppo_model = PPO("MlpPolicy", ppo_env, verbose=0)

In [ ]:
class RewardCallback(BaseCallback):
    def __init__(self):
        super().__init__()
        self.rewards = []

    def _on_step(self):
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.rewards.append(info["episode"]["r"])
        return True

callback = RewardCallback()

ppo_model.learn(total_timesteps=100000, callback=callback)

In [ ]:
ppo_rewards = callback.rewards

plt.plot(np.convolve(ppo_rewards, np.ones(100)/100, mode='valid'))
plt.title("PPO Average Reward")
plt.xlabel("Episodes")
plt.ylabel("Reward")
plt.show()

In [ ]:
successes = 0
test_episodes = 1000

for _ in range(test_episodes):
    state, _ = ppo_env.reset()
    done = False

    while not done:
        action, _ = ppo_model.predict(state, deterministic=True)
        action = int(action)

        state, reward, terminated, truncated, _ = ppo_env.step(action)
        done = terminated or truncated

    if reward > 0:
        successes += 1

print("PPO Success rate:", successes / test_episodes)

# Results

Final runs used about 5,000 training episodes for each agent (PPO: 100,000 timesteps).

| | DQN | PPO |
|---|---|---|
| Success rate in testing | 74.2% | 74.2% |
| Training time | about 22 minutes | about 2 minutes |
| Learning curve | rises quickly, then stays noisy | slow start, then steady and smooth |

74.2% matches the best possible success rate on this map. Dynamic programming over FrozenLake's slippery transition probabilities gives a maximum of 74.22% within Gymnasium's 100-step limit, so the 4x4 map is too easy to separate the two algorithms on success rate.

Outputs are cleared in this copy; re-run the cells to reproduce. Exact numbers vary from run to run. Note that the DQN reward curve includes the shaping penalties while the PPO curve uses the environment's raw reward, so the curves are not on the same scale.